In [ ]:
import os
import sys

In [ ]:
!pip uninstall -y numpy
!pip install numpy==2.0.2

In [ ]:
!pip install pyarrow
!pip install matplotlib
!pip install rich
!pip install protobuf

In [ ]:
import os
import torch

def detect_environment():
    # Check for GPU
    has_gpu = torch.cuda.is_available()

    # Check for Colab-specific environment
    is_colab = 'COLAB_GPU' in os.environ or 'google.colab' in str(get_ipython())

    # Check for RunPod-specific environment
    is_runpod = 'RUNPOD_POD_ID' in os.environ or os.path.exists('/workspace')

    if is_runpod and has_gpu:
        return 'runpod'
    elif is_colab and not has_gpu:
        return 'colab'
    else:
        return 'unknown'


In [ ]:
from pathlib import Path

#check the environement
env = detect_environment()
print("env", env)
if env == 'runpod':
  ##rclone sync gdrive:/MyDrive/MLProjects/foundation-models-radiology /workspace/MLProjects/foundation-models-radiology
  ROOT = Path('/workspace/MLProjects/foundation-models-radiology')
elif env == 'colab':
  from google.colab import drive
  drive.mount('/content/drive')
  ###once mounted the folders can be referenced
  ROOT = Path('/content/drive/MyDrive/MLProjects/foundation-models-radiology')

else:
  sys.exit("Error: No platform recognised")

DICOM_DIR = ROOT / 'PTXHeadtoHeadSmall'   # use the exact folder name as on Drive
JPEG_DIR = ROOT / 'cxr_jpegs'
JPEG_DIR.mkdir(exist_ok=True)
print("exists:", ROOT.exists())


In [ ]:
from zipfile import ZipFile
from pathlib import Path

DEST = Path("/workspace/MLProjects")
zip_path = next(DEST.glob("*.zip"))  # first zip in the folder
with ZipFile(zip_path) as zf:
    zf.extractall(DEST)
zip_path.unlink()  # delete the zip

In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image
import pydicom
from pydicom.pixel_data_handlers.util import apply_voi_lut

def dcm_to_rgb_pil(ds):
    """Return a PIL.Image in RGB suitable for CheXagent."""
    # Decode pixel data (uses pylibjpeg/gdcm if installed)
    arr = apply_voi_lut(ds.pixel_array, ds) if hasattr(ds, "PixelData") else None
    if arr is None:
        raise RuntimeError("No pixel data")

    # If signed, shift to positive
    if ds.get("PixelRepresentation", 0) == 1:  # signed
        # Typical CT; for CXR it’s often unsigned, but this is safe
        arr = arr.astype(np.int32)
    
    # Handle MONOCHROME1 inversion (white=low)
    photo = str(ds.get("PhotometricInterpretation", "")).upper()
    if photo == "MONOCHROME1":
        arr = arr.max() - arr

    # Normalize to 0–255 uint8
    arr = arr.astype(np.float32)
    # Prefer DICOM windowing; if no VOI, do a gentle percentile clip
    if not hasattr(ds, "WindowCenter") or not hasattr(ds, "WindowWidth"):
        lo, hi = np.percentile(arr, [0.5, 99.5])
        if hi <= lo:  # degenerate
            lo, hi = float(arr.min()), float(arr.max())
        arr = np.clip((arr - lo) / max(hi - lo, 1e-6), 0, 1)
    else:
        # apply_voi_lut already used above; just min-max in case values are not 0..255
        mn, mx = float(arr.min()), float(arr.max())
        arr = (arr - mn) / max(mx - mn, 1e-6)

    arr = (arr * 255.0 + 0.5).astype(np.uint8)

    # Grayscale -> RGB (3-channel) for CheXagent
    if arr.ndim == 2:
        img = Image.fromarray(arr, mode="L").convert("RGB")
    elif arr.ndim == 3 and arr.shape[-1] in (3, 4):
        # Color DICOMs (rare for CXR); drop alpha if present
        img = Image.fromarray(arr[..., :3], mode="RGB")
    else:
        raise RuntimeError(f"Unexpected array shape {arr.shape}")
    return img

def convert_dicom_to_jpegs(
    src_dir, dst_dir, recursive=True, resize_short=None, quality=95, save_png=False
):
    src_dir, dst_dir = Path(src_dir), Path(dst_dir)
    dst_dir.mkdir(parents=True, exist_ok=True)
    patterns = ["*.dcm", "*.DCM", "*"] if recursive else ["*.dcm", "*.DCM"]
    dicoms = []
    for pat in patterns:
        dicoms += (src_dir.rglob(pat) if recursive else src_dir.glob(pat))
    processed = errors = 0
    for p in sorted(set(dicoms)):
        try:
            # quick DICOM sniff: files without extension may still be DICOM
            ds = pydicom.dcmread(p, force=True)
            img = dcm_to_rgb_pil(ds)
            # Optional: resize by short side while keeping aspect ratio
            if resize_short:
                w, h = img.size
                s = min(w, h)
                if s != resize_short:
                    scale = resize_short / s
                    img = img.resize((int(w*scale), int(h*scale)), Image.BICUBIC)
            # Multi-frame?
            nframes = int(getattr(ds, "NumberOfFrames", 1) or 1)
            stem = p.stem
            if nframes > 1:
                # Save first few frames; adjust if you want all frames
                for i in range(min(nframes, 3)):
                    frame = dcm_to_rgb_pil(ds[i]) if hasattr(ds, "__getitem__") else img
                    out = dst_dir / f"{stem}_f{i}.{'png' if save_png else 'jpg'}"
                    _save_img(frame, out, quality, save_png)
            else:
                out = dst_dir / f"{stem}.{'png' if save_png else 'jpg'}"
                _save_img(img, out, quality, save_png)
            processed += 1
        except Exception as e:
            # Comment this print if too chatty
            print(f"[warn] {p}: {e}")
            errors += 1
    print(f"[done] converted: {processed}, errors: {errors}, out -> {dst_dir}")

def _save_img(img: Image.Image, path: Path, quality=95, png=False):
    path.parent.mkdir(parents=True, exist_ok=True)
    if png:
        img.save(path, format="PNG", optimize=True)
    else:
        img.save(path, format="JPEG", quality=quality, optimize=True)

# ---- Example call (your paths)
# Source DICOMs here:
src = "/workspace/MLProjects/PTXHeadtoHeadSmall"
# JPEGs out here:
dst = "/workspace/MLProjects/PTXHeadtoHeadSmall/cxr_jpegs"
convert_dicom_to_jpegs(src, dst, recursive=True, resize_short=1024, quality=95, save_png=False)


In [ ]:
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM
from PIL import Image
import warnings, re
import requests
from datetime import datetime
from io import BytesIO

# ---- Put cache/temp on /workspace (adjust if needed)
os.environ.setdefault("HF_HOME", "/workspace/.hf")
os.makedirs(os.environ["HF_HOME"], exist_ok=True)
os.environ.setdefault("TRANSFORMERS_CACHE", "/workspace/.hf/transformers")
os.makedirs(os.environ["TRANSFORMERS_CACHE"], exist_ok=True)
os.environ.setdefault("TMPDIR", "/workspace/tmp")
os.makedirs(os.environ["TMPDIR"], exist_ok=True)
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")

# ---- Small helpers
def _stamp(t0=None):
    wall = datetime.now().strftime("%H:%M:%S.%f")[:-3]
    if t0 is None:
        return f"[{wall}]"
    return f"[{wall} +{time.perf_counter()-t0:.2f}s]"

def load_image_from_url(url, timeout=(10, 60)):
    r = requests.get(url, timeout=timeout); r.raise_for_status()
    return Image.open(BytesIO(r.content)).convert("RGB")
    
class CheXagent(object):
    def __init__(self):
        self.t0 = time.perf_counter()

        # step 1: Setup constant
        self.model_name = "StanfordAIMI/CheXagent-2-3b"
        self.device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.dtype      = torch.bfloat16 if self.device.type == "cuda" else torch.float32
        device_map      = "cuda:0" if self.device.type == "cuda" else None
        ###self.revision is the exact snapshot of the model repo on Hugging Face that you want to load. 
        ####You pass it to from_pretrained(..., revision=self.revision) so Transformers fetches the files (weights + “remote code” like modeling_chexagent.py, tokenization_chexagent.py) from that specific commit/tag/branch.
        self.revision   = "463999422d77fe01380ef03493e5ae3bb11bd004"  # pin remote code
                
        # step 2: Load Processor and Model
        print(_stamp(self.t0), "loading tokenizer…", flush=True)
        
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.model_name, trust_remote_code=True, revision=self.revision,
            cache_dir=os.environ.get("TRANSFORMERS_CACHE")
        )
        print(_stamp(self.t0), "loading model (this can take a while first run)…", flush=True)
        
        # 1) Load on CPU in fp32 (safe), no sharding
        base = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            trust_remote_code=True, revision=self.revision,
            cache_dir=os.environ.get("TRANSFORMERS_CACHE"),
            device_map=None,
            low_cpu_mem_usage=False,
            torch_dtype=torch.float32
        )
        # 2) If CUDA is available, move + cast once to CUDA bf16
        if self.device.type == "cuda":
            self.model = base.to(self.device, dtype=torch.bfloat16)
            torch.backends.cuda.matmul.allow_tf32 = True
        else:            
            self.model = base # CPU-only: keep fp32

        self.model.eval()
        print(_stamp(self.t0), "model ready.", flush=True)
        
    def generate(self, paths, prompt):
        t0 = time.time()
        print("[step 1] before tokenizer", flush=True)
        query = self.tokenizer.from_list_format(
            [*({'image': p} for p in paths), {'text': prompt}]
        )
        print("[step 2] after tokenizer, before apply_chat_template", flush=True)
        
        ##added .to(self.device) find out what difference
        conv = [{"from": "system", "value": "You are a helpful assistant."}, {"from": "human", "value": query}]
        input_ids = self.tokenizer.apply_chat_template(
            conv, add_generation_prompt=True, return_tensors="pt"
        ).to(self.device)

        attention_mask = torch.ones_like(input_ids, dtype=torch.long, device=self.device)
        
        output = self.model.generate(
            input_ids.to(self.device), attention_mask=attention_mask, do_sample=False, num_beams=1, temperature=1., top_p=1., use_cache=True,
            max_new_tokens=512
        )[0]
        print("[step 3] after appy_chat_template", flush=True)
        response = self.tokenizer.decode(output, skip_special_tokens=True)
        ##original - but prev. a problem -> response = self.tokenizer.decode(output[input_ids.size(1):-1])
        return response

    def view_classification(self, path):
        assert isinstance(path, str)
        prompt = "What is the view of this chest X-ray? Options: (a) PA, (b) AP, (c) LATERAL"
        response = self.generate([path], prompt)
        return response

    ##passing src
    def binary_disease_classification(self, src, disease_name, max_new_tokens=256, max_images=3):
        if isinstance(src, (str, Path)) and Path(src).is_dir():
            files = []
            for ext in ("*.jpg","*.jpeg","*.png","*.bmp","*.tif","*.tiff"):
                files += sorted(Path(src).glob(ext))
                paths = [str(p) for p in files[:max_images]]
        elif isinstance(src, (list, tuple)):
            paths = [str(p) for p in src][:max_images]
        else:
            raise ValueError("src must be a directory or a list of image paths")

        prompt = f'Does this chest X-ray contain a {disease_name}?'
        results = []
        for i, p in enumerate(paths[:max_images], 1):
            ans = self.generate([p], prompt)
            results.append({"picture": i, "image": p, "answer": ans})
        return results

        



In [ ]:
from rich import print
# Make sure caches & temp are on /workspace, and disable hf-xet just in case

def main():
    # Load the model
    os.environ["HF_HOME"] = "/workspace/.hf"
    os.environ["TRANSFORMERS_CACHE"] = "/workspace/.hf/transformers"
    os.environ["TMPDIR"] = "/workspace/tmp"
    os.environ["HF_HUB_DISABLE_XET"] = "1"
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

    chexagent = CheXagent()
    try:
        # Task 3: Binary Disease Classification
        #path = "/workspace/MLProjects/PTXHeadtoHeadSmall/cxr_jpegs"
        #response = chexagent.binary_disease_classification(path, "Pneumothorax")
        resp = chexagent.binary_disease_classification(
            "/workspace/MLProjects/PTXHeadtoHeadSmall/cxr_jpegs", "Pneumothorax", max_images=2
        )
        for r in resp:
            print(r["picture"], r["answer"])  

    except Exception as e:
        print("[ERROR]", repr(e))
        traceback.print_exc()   # <— show the real stack trace
        raise                   # <— let it surface

if __name__ == '__main__':
    main()